In [ ]:
import random, os
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizerFast, BertModel
from sklearn.linear_model import LogisticRegression

In [ ]:
seed = 2026
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
BASE = '/kaggle/input/competitions/polarity-aicc-round-7'

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')

print(f"train {train.shape}, test {test.shape}, label counts {train['label'].value_counts().to_dict()}")
train.head()

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-large-uncased')
bert = BertModel.from_pretrained('bert-large-uncased').eval().to(device)
    
embeddings = bert.get_input_embeddings().weight  

In [ ]:
def word_vector(word):
    return embeddings[tokenizer.convert_tokens_to_ids(word)]

def pair_features(a, b):
    va, vb = word_vector(a), word_vector(b)
    cosine = torch.cosine_similarity(va, vb, dim=0).item()
    distance = (va - vb).norm().item()
    return [cosine, distance]

X_train = np.array([pair_features(a, b) for a, b in zip(train['w1'], train['w2'])])
X_test = np.array([pair_features(a, b) for a, b in zip(test['w1'], test['w2'])])
y_train = train['label'].values

In [ ]:
mean, std = X_train.mean(0), X_train.std(0) 
X_train, X_test = (X_train - mean) / std, (X_test - mean) / std

clf = LogisticRegression().fit(X_train, y_train)

print(f"train accuracy {(clf.predict(X_train) == y_train).mean():.3f}")

In [ ]:
pred = clf.predict(X_test)

submission = pd.DataFrame({'row_id': test['row_id'], 'label': pred})
submission.to_csv('submission.csv', index=False)

In [ ]:
print(f"{int(pred.sum())} antonyms predicted of {len(pred)}")
submission.head()